# 🏥 Dashboard — Cartographie des infrastructures sanitaires de Djibouti

Source : chapitre III de l'analyse *Santé* (données RGPH-3 / cartographie censitaire).

**Mode d'emploi (Google Colab)** : menu *Exécution → Tout exécuter*. Les graphiques sont interactifs (survol, zoom, menus déroulants).

**Important** : exécutez toujours la cellule 1 en premier (ou utilisez *Tout exécuter*) : elle charge les données utilisées par toutes les autres.

**Avant de commencer** : modifiez `GITHUB_USER` et `GITHUB_REPO` dans la cellule 1 une fois le dépôt publié. Tant que le dépôt n'est pas en ligne, importez le dossier `data/` dans l'onglet *Fichiers* de Colab.

In [ ]:
# 1. Configuration : remplacez par votre dépôt GitHub
GITHUB_USER = "VOTRE-UTILISATEUR"
GITHUB_REPO = "dashboard-sante-djibouti"
BRANCH = "main"

import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML

RAW = "https://raw.githubusercontent.com/" + GITHUB_USER + "/" + GITHUB_REPO + "/" + BRANCH + "/data/"

def charger(nom):
    """Lit d'abord data/ (local ou Colab), sinon le dépôt GitHub."""
    local = Path("data") / nom
    if local.exists():
        return pd.read_csv(local)
    try:
        return pd.read_csv(RAW + nom)
    except Exception:
        raise FileNotFoundError(
            "Fichier introuvable : " + nom + ". Deux solutions : (1) importez le dossier data/ dans l'onglet "
            "Fichiers de Colab (icône dossier à gauche), ou (2) renseignez GITHUB_USER / GITHUB_REPO "
            "avec votre dépôt GitHub public, puis relancez cette cellule.")

ORDRE = ["Djibouti-Ville", "Ali Sabieh", "Dikhil", "Tadjourah", "Obock", "Arta"]
PALETTE = ["#1D6FB8", "#1D9E75", "#E8943A", "#7F77DD", "#D4537E", "#888780"]
COUL_SECTEUR = {"Public": "#1D6FB8", "Privé": "#E8943A"}
FIGS = {}

def montrer(cle, fig, hauteur=450, afficher=True):
    fig.update_layout(template="plotly_white", height=hauteur,
                      margin=dict(l=40, r=20, t=70, b=40),
                      font=dict(family="Arial", size=13))
    FIGS[cle] = fig
    if afficher:
        fig.show()

# Chargement des données (même cellule : tout est prêt dès qu'elle est exécutée)
niveaux = charger("niveaux.csv")
regions = charger("regions.csv")
types = charger("infra_par_type_region.csv")
fonct = charger("fonctionnalite_region.csv")
secteur = charger("secteur_region.csv")
raisons = charger("raisons_non_fonctionnalite.csv")
capital = charger("djibouti_ville_communes.csv")
sp = charger("sous_prefectures.csv")
print("Données chargées :", len(types), "lignes (types) |", len(sp), "sous-préfectures")


## 1. Chiffres clés

In [ ]:
total = int(fonct["total"].sum())
ok = int(fonct["fonctionnelles"].sum())
prive = int(secteur["prive"].sum())
dv = int(fonct.loc[fonct.region == "Djibouti-Ville", "total"].iloc[0])
lits_dv = int(capital["lits"].sum())
infirm_dv = int(capital["infirmiers"].sum())
accou_dv = int(capital["accoucheuses"].sum())

def carte_kpi(titre, valeur, sous=""):
    return ("<div style='flex:1;min-width:150px;padding:14px 16px;border-radius:10px;"
            "background:#f4f6fa;border:1px solid #e3e7ef;font-family:Arial,sans-serif'>"
            "<div style='font-size:12px;color:#556'>" + titre + "</div>"
            "<div style='font-size:28px;font-weight:600;color:#1a2b4c'>" + str(valeur) + "</div>"
            "<div style='font-size:12px;color:#667'>" + sous + "</div></div>")

KPI_HTML = ("<div style='display:flex;gap:12px;flex-wrap:wrap'>"
    + carte_kpi("Infrastructures recensées", total, "6 régions")
    + carte_kpi("Fonctionnelles", str(round(ok / total * 100, 1)) + " %", str(ok) + " sur " + str(total))
    + carte_kpi("Secteur privé", str(round(prive / total * 100, 1)) + " %", str(prive) + " infrastructures")
    + carte_kpi("Concentration Djibouti-Ville", str(round(dv / total * 100, 1)) + " %", str(dv) + " infrastructures")
    + carte_kpi("Lits (Djibouti-Ville)", f"{lits_dv:,}".replace(",", " "), "3 communes")
    + carte_kpi("Infirmiers (Djibouti-Ville)", f"{infirm_dv:,}".replace(",", " "), str(accou_dv) + " accoucheuses")
    + "</div>")
display(HTML(KPI_HTML))

## 2. Répartition des infrastructures

In [ ]:
# 2.1 Niveaux de la pyramide sanitaire
fig = px.pie(niveaux, names="niveau", values="nombre", hole=0.55,
             color_discrete_sequence=PALETTE,
             title="Pyramide sanitaire : " + str(int(niveaux.nombre.sum())) + " infrastructures classées sur 190")
fig.update_traces(textinfo="label+percent+value")
montrer("niveaux", fig, 420)

# 2.2 Par région
par_region = (types.groupby("region")["nombre"].sum().reindex(ORDRE).reset_index())
par_region["part_pct"] = (par_region["nombre"] / par_region["nombre"].sum() * 100).round(1)
fig = px.bar(par_region, x="region", y="nombre", text="nombre", hover_data=["part_pct"],
             color_discrete_sequence=[PALETTE[0]], title="Nombre d'infrastructures par région",
             labels={"region": "", "nombre": "Infrastructures"})
montrer("regions", fig, 420)

In [ ]:
# 2.3 Types d'infrastructures x régions (carte de chaleur)
pivot = (types.pivot_table(index="type", columns="region", values="nombre", aggfunc="sum")
         .reindex(index=types["type"].unique(), columns=ORDRE))
fig = px.imshow(pivot, text_auto=True, aspect="auto", color_continuous_scale="Blues",
                title="Types d'infrastructures par région")
montrer("heatmap", fig, 620)

In [ ]:
# 2.4 Détail interactif : choisissez une région
def types_region(region="Toutes", afficher=True):
    d = types if region == "Toutes" else types[types.region == region]
    d = d.groupby("type", as_index=False)["nombre"].sum()
    d = d[d.nombre > 0].sort_values("nombre")
    fig = px.bar(d, x="nombre", y="type", orientation="h", text="nombre",
                 color_discrete_sequence=[PALETTE[1]],
                 title="Types d'infrastructures : " + region,
                 labels={"type": "", "nombre": "Nombre"})
    montrer("types_" + region, fig, max(350, 30 * len(d) + 130), afficher)

widgets.interact(types_region,
                 region=widgets.Dropdown(options=["Toutes"] + ORDRE, value="Toutes", description="Région :"),
                 afficher=widgets.fixed(True));

## 3. Fonctionnement et raisons d'inactivité

In [ ]:
f = fonct.set_index("region").reindex(ORDRE).reset_index()
f["taux"] = (f["fonctionnelles"] / f["total"] * 100).round(1)

fig = go.Figure()
fig.add_bar(x=f["region"], y=f["fonctionnelles"], name="Fonctionnelles", marker_color="#1D9E75",
            text=f["fonctionnelles"], textposition="inside")
fig.add_bar(x=f["region"], y=f["non_fonctionnelles"], name="Non fonctionnelles", marker_color="#D4537E",
            text=f["non_fonctionnelles"], textposition="inside")
fig.update_layout(barmode="stack", title="Infrastructures fonctionnelles et non fonctionnelles")
montrer("fonctionnement", fig, 420)

fig = px.bar(f, x="region", y="taux", text="taux", color_discrete_sequence=[PALETTE[0]],
             title="Taux de fonctionnalité (%)", labels={"region": "", "taux": "%"})
fig.update_yaxes(range=[0, 105])
montrer("taux", fig, 380)

In [ ]:
# Raisons de non-fonctionnalité
lib = {"construction": "En construction", "rehabilitation": "En réhabilitation",
       "ruine": "En ruine", "manque_personnel": "Manque de personnel soignant"}
r = raisons.melt(id_vars="region", var_name="raison", value_name="nombre")
r["raison"] = r["raison"].map(lib)
fig = px.bar(r, x="region", y="nombre", color="raison", barmode="stack",
             category_orders={"region": ORDRE}, color_discrete_sequence=PALETTE,
             title="Raisons de non-fonctionnalité (22 infrastructures)",
             labels={"region": "", "nombre": "Infrastructures"})
montrer("raisons", fig, 420)

bilan = r.groupby("raison", as_index=False)["nombre"].sum().sort_values("nombre", ascending=False)
display(bilan)

## 4. Public / privé

In [ ]:
sec = secteur.melt(id_vars=["region", "total"], value_vars=["public", "prive"],
                   var_name="secteur", value_name="nombre")
sec["secteur"] = sec["secteur"].map({"public": "Public", "prive": "Privé"})
fig = px.bar(sec, x="region", y="nombre", color="secteur", text="nombre", barmode="stack",
             category_orders={"region": ORDRE}, color_discrete_map=COUL_SECTEUR,
             title="Part du public et du privé par région (en %)",
             labels={"region": "", "nombre": "Infrastructures"})
fig.update_layout(barnorm="percent", yaxis_title="% des infrastructures")
montrer("secteur", fig, 440)

## 5. Djibouti-Ville : lits, accoucheuses et infirmiers par commune

In [ ]:
INDIC = {"lits": "Lits", "accoucheuses": "Accoucheuses", "infirmiers": "Infirmiers"}

def capitale(indicateur="lits", afficher=True):
    fig = px.bar(capital, x="commune", y=indicateur, color="niveau", barmode="group", text=indicateur,
                 color_discrete_sequence=PALETTE,
                 title="Djibouti-Ville : " + INDIC[indicateur] + " par commune et par niveau",
                 labels={"commune": "", indicateur: INDIC[indicateur], "niveau": "Niveau"})
    fig.update_layout(legend=dict(orientation="h", y=-0.2))
    montrer("capitale_" + indicateur, fig, 480, afficher)

widgets.interact(capitale,
                 indicateur=widgets.Dropdown(options=[(v, k) for k, v in INDIC.items()], value="lits",
                                             description="Indicateur :"),
                 afficher=widgets.fixed(True));

## 6. Les autres régions : sous-préfectures

In [ ]:
fig = px.treemap(sp, path=["region", "sous_prefecture"], values="total_infrastructures",
                 color="region", color_discrete_sequence=PALETTE,
                 title="Infrastructures par région et sous-préfecture (hors Djibouti-Ville)")
montrer("treemap", fig, 480)

In [ ]:
def region_detail(region="Ali Sabieh", afficher=True):
    d = sp[sp.region == region]
    # (a) Ressources
    lg = d.melt(id_vars="sous_prefecture", value_vars=["lits", "accoucheuses", "infirmiers"],
                var_name="ressource", value_name="valeur")
    lg["ressource"] = lg["ressource"].map(INDIC)
    fig = px.bar(lg, x="sous_prefecture", y="valeur", color="ressource", barmode="group", text="valeur",
                 color_discrete_sequence=PALETTE, title=region + " : ressources par sous-préfecture",
                 labels={"sous_prefecture": "", "valeur": "Effectif", "ressource": ""})
    montrer("ressources_" + region, fig, 420, afficher)
    # (b) Types
    cols = {"hopitaux_reference": "Hôpitaux et centres de référence",
            "centres_medico_hospitaliers": "Centres médico-hospitaliers",
            "centres_sante_communautaires": "Centres de santé communautaires et parapublics"}
    tg = d.melt(id_vars="sous_prefecture", value_vars=list(cols), var_name="type", value_name="nombre")
    tg["type"] = tg["type"].map(cols)
    fig = px.bar(tg, x="sous_prefecture", y="nombre", color="type", barmode="stack", text="nombre",
                 color_discrete_sequence=PALETTE, title=region + " : infrastructures par type",
                 labels={"sous_prefecture": "", "nombre": "Infrastructures", "type": ""})
    fig.update_layout(legend=dict(orientation="h", y=-0.25))
    montrer("types_sp_" + region, fig, 450, afficher)
    if afficher:
        display(d.set_index("sous_prefecture")[["total_infrastructures", "lits", "accoucheuses", "infirmiers"]])

widgets.interact(region_detail,
                 region=widgets.Dropdown(options=ORDRE[1:], value="Ali Sabieh", description="Région :"),
                 afficher=widgets.fixed(True));

## 7. Concentration des ressources et carte

In [ ]:
res = sp.groupby("region")[["lits", "accoucheuses", "infirmiers"]].sum()
res.loc["Djibouti-Ville"] = capital[["lits", "accoucheuses", "infirmiers"]].sum()
res = res.reindex(ORDRE)
res.index.name = "region"
part = (res / res.sum() * 100).round(1).reset_index()
pl = part.melt(id_vars="region", var_name="ressource", value_name="part_pct")
pl["ressource"] = pl["ressource"].map(INDIC)
fig = px.bar(pl, x="region", y="part_pct", color="ressource", barmode="group", text="part_pct",
             color_discrete_sequence=PALETTE, title="Part de chaque région dans les ressources nationales recensées (%)",
             labels={"region": "", "part_pct": "% du total", "ressource": ""})
montrer("concentration", fig, 450)
display(res)

In [ ]:
ff = f[["region", "total", "fonctionnelles", "taux"]].rename(columns={"taux": "taux_fonct"})
carte = regions.merge(ff, on="region").merge(res.reset_index(), on="region")
try:
    fig = px.scatter_map(carte, lat="lat", lon="lon", size="total", color="taux_fonct",
                         hover_name="region", hover_data=["fonctionnelles", "lits", "infirmiers"],
                         size_max=45, zoom=7, color_continuous_scale="RdYlGn",
                         map_style="open-street-map", title="Carte : infrastructures et taux de fonctionnalité")
except AttributeError:
    fig = px.scatter_mapbox(carte, lat="lat", lon="lon", size="total", color="taux_fonct",
                            hover_name="region", hover_data=["fonctionnelles", "lits", "infirmiers"],
                            size_max=45, zoom=7, color_continuous_scale="RdYlGn",
                            mapbox_style="open-street-map", title="Carte : infrastructures et taux de fonctionnalité")
montrer("carte", fig, 560)

## 8. Contrôles qualité des données
Chaque total du document est recalculé à partir des tableaux détaillés.

In [ ]:
controles = []
def check(nom, attendu, obtenu):
    controles.append({"Contrôle": nom, "Valeur du document": attendu,
                      "Valeur recalculée": int(obtenu), "Cohérent": "✅" if attendu == int(obtenu) else "⚠️"})

check("Total infrastructures (tableau par type)", 190, types["nombre"].sum())
check("Niveaux I + II + III", 190, niveaux["nombre"].sum())
check("Infrastructures fonctionnelles", 168, fonct["fonctionnelles"].sum())
check("Infrastructures non fonctionnelles", 22, fonct["non_fonctionnelles"].sum())
check("Secteur public", 93, secteur["public"].sum())
check("Secteur privé", 97, secteur["prive"].sum())
check("Raisons de non-fonctionnalité", 22, raisons.drop(columns="region").sum().sum())
check("Lits Djibouti-Ville", 1243, capital["lits"].sum())
check("Accoucheuses Djibouti-Ville", 81, capital["accoucheuses"].sum())
check("Infirmiers Djibouti-Ville", 1002, capital["infirmiers"].sum())
check("Fonctionnelles hors Djibouti-Ville (texte : 54)", 54,
      fonct.loc[fonct.region != "Djibouti-Ville", "fonctionnelles"].sum())
display(pd.DataFrame(controles))

### Points à corriger dans le document Word

| # | Constat | Correction proposée |
|---|---|---|
| 1 | Niveaux : 128 + 54 + 6 = **188**, alors que le total est 190 | Ajouter les 2 infrastructures manquantes (ex. hôpitaux généraux de référence) ou préciser « non classées » |
| 2 | Tableau par type : cellules tronquées (`2-`, `5-`, `1-`, `4-`) | Valeurs reconstituées : **20** centres de santé, **50** postes de santé, **10** centres médicaux, **40** pharmacies à Djibouti-Ville |
| 3 | Texte : « 54 fonctionnelles dans les autres régions » | 16 + 11 + 17 + 11 + 10 = **65** |
| 4 | Tableau des communes de Djibouti-Ville : les % (67,2 / 25,0 / 30,5 / 11,7) sont en réalité les **parts de lits** dans le total national (ex. 462 / 1 849 = 25,0 %), pas des infrastructures. Le texte (46 % / 35 % / 20 %) ne correspond ni à ces valeurs ni aux parts de lits calculées dans la capitale (45,4 % / 37,2 % / 17,5 %) | Renommer le tableau « part des lits » et fournir les **effectifs** d'infrastructures par commune |
| 5 | Colonne « Ensemble » des tableaux de personnel = lits + accoucheuses + infirmiers, présentée comme « employés » | Additionner uniquement accoucheuses + infirmiers, ou renommer la colonne |
| 6 | Texte Tadjourah : lits « Adaillou 56, Dorra 46, Randa 10 » | Tableau : Adaillou **46**, Dorra **23**, Randa **8** (56 = « ensemble » d'Adaillou) |
| 7 | Texte accoucheuses : « aucune à Balbala et Ras-Dika » | Vrai uniquement pour les hôpitaux de référence (Balbala 19, Ras-Dika 6 au total) |

## 9. Export : dashboard HTML autonome
Génère `dashboard.html` (un seul fichier, interactif) à publier sur GitHub Pages ou à envoyer par e-mail.

In [ ]:
for ind in INDIC:
    capitale(ind, afficher=False)

EXPORT = ["niveaux", "regions", "heatmap", "fonctionnement", "taux", "raisons", "secteur",
          "capitale_lits", "capitale_accoucheuses", "capitale_infirmiers", "treemap",
          "concentration", "carte"]
blocs = []
for cle in EXPORT:
    if cle in FIGS:
        inclure_js = "cdn" if not blocs else False
        blocs.append("<section class='bloc'>" + FIGS[cle].to_html(full_html=False, include_plotlyjs=inclure_js) + "</section>")

page = ("<!DOCTYPE html><html lang='fr'><head><meta charset='utf-8'>"
        "<meta name='viewport' content='width=device-width, initial-scale=1'>"
        "<title>Infrastructures sanitaires de Djibouti</title>"
        "<style>body{font-family:Arial,sans-serif;max-width:1100px;margin:0 auto;padding:24px;background:#fff;color:#1a2b4c}"
        ".bloc{margin:28px 0;border:1px solid #e3e7ef;border-radius:10px;padding:8px}</style></head><body>"
        "<h1>Cartographie des infrastructures sanitaires de Djibouti</h1>" + KPI_HTML + "".join(blocs) + "</body></html>")
Path("dashboard.html").write_text(page, encoding="utf-8")
print("dashboard.html créé :", len(page) // 1024, "Ko")

try:
    from google.colab import files
    files.download("dashboard.html")
except ImportError:
    pass